# Studio di MEREGE
basato su cleaning 4
## Inizializzazione ed Import

In [1]:
from data_model.DataCleaner import *
from dl_client import DatalakeClient
from tests_utils.manage_excel_support_file import *
import pandas as pd
import os

client = DatalakeClient()

# Download the raw files 
Exclusevily from ADNI dataset stored in the Datalake

In [2]:
file_codes = ['UCSFFSX', 'UCSFFSX51', 'UCSFFSX6', 'UCSFFSX7', 'UCSFFSL', 'UCSDVOL', 'UPENN_ROI_MARS']

#'ADNIMERGE', 'PTDEMOG', 'DXSUM', 'MMSE', 'ADAS', 'FAQ', 'CDR', 'MOCA', 'APOERES', 
#            'UCSFFSX', 'UCSFFSX51', 'UCSFFSX6', 'UCSFFSX7', 'UCSFFSX51_ADNI1_3T', 'UCSFFSL51ALL', 'UCSFFSL51', 'UCSFFSL51Y1', 'UCSFFSL', 'UCSDVOL', 'UPENN_ROI_MARS'

### Mixed info ###
# 'ADNIMERGE', 
# 'ADSP_PHC_BIOMARKER', 'ADNI_DIAN_COMPARISON', --> have CSF

### Single Cofactor ###
# 'PTDEMOG', 'DXSUM', 'MMSE', 'ADAS', 'FAQ', 'CDR', 'MOCA', 'APOERES'

### Volumes ###
# 'UCSFFSX', 'UCSFFSX51', 'UCSFFSX6', 'UCSFFSX7', 'UCSFFSL', 'UCSDVOL', 'UPENN_ROI_MARS',
# 'UCSFFSL51ALL', 'UCSFFSL51', 'UCSFFSL51Y1', 'UCSFFSX51_ADNI1_3T'  --> just partial immages segmentation

### CSF ###
# 'UPENNBIOMK_ADNIDIAN_ES_2017', 'UPENNBIOMK_ROCHE_ELECSYS', 'EUROIMMUN', 'FUJIREBIOABETA', 'SALADAX_BIOMEDICAL', 'MESOSCALE', 'UPENNBIOMK_MASTER', 'UPENN_2DUPLC_CRM', 

In [3]:
search = client.query_files(
    query={'custom.level' : 'cleaned_04', 'custom.source' : 'ADNI', 'custom.file_code': file_codes})

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True)


In [4]:
len(zip_files)

7

In [5]:
file_name_0 = list(zip_files.keys())[0]
df_0 = zip_files[file_name_0].copy(deep=True)

file_name_1 = list(zip_files.keys())[1]
df_1 = zip_files[file_name_1].copy(deep=True)


file_name_2 = list(zip_files.keys())[2]
df_2 = zip_files[file_name_2].copy(deep=True)

file_name_3 = list(zip_files.keys())[3]
df_3 = zip_files[file_name_3].copy(deep=True)

file_name_4 = list(zip_files.keys())[4]
df_4 = zip_files[file_name_4].copy(deep=True)

file_name_5 = list(zip_files.keys())[5]
df_5 = zip_files[file_name_5].copy(deep=True)

file_name_6 = list(zip_files.keys())[6]
df_6 = zip_files[file_name_6].copy(deep=True)

'''
file_name_7 = list(zip_files.keys())[7]
df_7 = zip_files[file_name_7].copy(deep=True)
'''
for x in range(len(zip_files)):
    print(x, '--->', list(zip_files.keys())[x])


0 ---> UCSFFSX_11_02_15_11Aug2025_04.csv
1 ---> UCSFFSX7_11Aug2025_04.csv
2 ---> UCSFFSX6_11Aug2025_04.csv
3 ---> UCSFFSX51_11_08_19_11Aug2025_04.csv
4 ---> UCSFFSL_02_01_16_11Aug2025_04.csv
5 ---> UPENNROI_MARS_06_01_16_09Oct2025_04.csv
6 ---> UCSDVOL_28Oct2025_04.csv


# Confronto stessi RID  ==> RID - EXAMDATE identici tra file

In [12]:
# Lista dei dataframe e nomi, per comodità
dfs = [df_0, df_1, df_2, df_3, df_4, df_5, df_6]
df_names = [file_name_0, file_name_1, file_name_2, file_name_3, file_name_4, file_name_5, file_name_6]
df_code = ['df_0', 'df_1', 'df_2', 'df_3', 'df_4', 'df_5', 'df_6']

# Assicurati che le colonne RID e EXAMDATE siano presenti e che EXAMDATE sia in formato datetime
for i, df in enumerate(dfs):
    if 'RID' not in df.columns:
        print(f"AVVISO: '{df_names[i]}' non contiene colonna RID")

# Matrice di match per (RID, EXAMDATE)
n = len(dfs)
match_matrix = pd.DataFrame(0, index=df_code, columns=df_code)

for i in range(n):
    for j in range(n):
        # Ci assicuriamo di confrontare solo se entrambe le colonne esistono nei dati
        if {'RID'}.issubset(dfs[i].columns) and {'RID'}.issubset(dfs[j].columns):
            s1 = set(dfs[i][['RID']].drop_duplicates().itertuples(index=False, name=None))
            s2 = set(dfs[j][['RID']].drop_duplicates().itertuples(index=False, name=None))
            match_matrix.iloc[i, j] = len(s1 & s2)
        else:
            match_matrix.iloc[i, j] = None  # Indica colonne mancanti


display_match_matrix = match_matrix.copy()
mask = np.triu(np.ones(display_match_matrix.shape), k=1).astype(bool)
display_match_matrix = display_match_matrix.mask(mask, "")

print("Matrice del numero di righe con stessi RID (solo diagonale inferiore):")
display(display_match_matrix)


Matrice del numero di righe con stessi RID (solo diagonale inferiore):


,df_0,df_1,df_2,df_3,df_4,df_5,df_6
df_0,844,,,,,,
df_1,11,807,,,,,
df_2,64,281,1122,,,,
df_3,82,73,318,1067,,,
df_4,755,11,63,82,755,,
df_5,840,11,64,82,753,840,
df_6,735,11,64,78,679,734,736


In [ ]:
# Lista dei dataframe e nomi, per comodità
dfs = [df_0, df_1, df_2, df_3, df_4, df_5, df_6]
df_names = [file_name_0, file_name_1, file_name_2, file_name_3, file_name_4, file_name_5, file_name_6]
df_code = ['df_0', 'df_1', 'df_2', 'df_3', 'df_4', 'df_5', 'df_6']

# Definisci il time_buffer (in giorni) per considerare match anche con date leggermente diverse
time_buffer = pd.Timedelta(days=0)  # Modifica questo valore secondo le tue esigenze (es. days=7 per una settimana)

# Assicurati che le colonne RID e EXAMDATE siano presenti e che EXAMDATE sia in formato datetime
for i, df in enumerate(dfs):
    if 'EXAMDATE' in df.columns:
        dfs[i]['EXAMDATE'] = pd.to_datetime(df['EXAMDATE'])
    else:
        print(f"AVVISO: '{df_names[i]}' non contiene colonna EXAMDATE")
    if 'RID' not in df.columns:
        print(f"AVVISO: '{df_names[i]}' non contiene colonna RID")

# Funzione helper per contare i match con buffer temporale
def count_matches_with_buffer(df1, df2, time_buffer):
    #### potremmo fare .py tipo MergerDf dove mettere queste funzioni
    """
    Conta il numero di match tra due dataframe considerando RID e EXAMDATE entro un buffer temporale.
    Due righe matchano se hanno lo stesso RID e |EXAMDATE1 - EXAMDATE2| <= time_buffer
    """
    # Prendi solo RID e EXAMDATE, rimuovi duplicati
    d1 = df1[['RID', 'EXAMDATE']].drop_duplicates()
    d2 = df2[['RID', 'EXAMDATE']].drop_duplicates()
    
    # Merge su RID per trovare tutti i possibili match
    merged = d1.merge(d2, on='RID', suffixes=('_1', '_2'))
    
    # Calcola la differenza assoluta tra le date
    merged['date_diff'] = (merged['EXAMDATE_1'] - merged['EXAMDATE_2']).abs()
    
    # Filtra per match entro il buffer temporale
    matches = merged[merged['date_diff'] <= time_buffer]
    
    # Conta i match unici (RID, EXAMDATE_1) - ogni riga di df1 può matchare con più righe di df2
    # ma vogliamo contare quante righe di df1 hanno almeno un match
    unique_matches = matches[['RID', 'EXAMDATE_1', 'EXAMDATE_2']].drop_duplicates()
    
    return unique_matches

# Matrice di match per (RID, EXAMDATE) con buffer temporale
n = len(dfs)
match_matrix = pd.DataFrame(0, index=df_code, columns=df_code)

for i in range(n):
    for j in range(n):
        # Ci assicuriamo di confrontare solo se entrambe le colonne esistono nei dati
        if {'RID', 'EXAMDATE'}.issubset(dfs[i].columns) and {'RID', 'EXAMDATE'}.issubset(dfs[j].columns):
            match_matrix.iloc[i, j] = len(count_matches_with_buffer(dfs[i], dfs[j], time_buffer))
        else:
            match_matrix.iloc[i, j] = None  # Indica colonne mancanti


display_match_matrix = match_matrix.copy()
mask = np.triu(np.ones(display_match_matrix.shape), k=1).astype(bool)
display_match_matrix = display_match_matrix.mask(mask, "")

print("Matrice del numero di righe con stessi RID (solo diagonale inferiore):")
display(display_match_matrix)




Matrice del numero di righe con stessi RID (solo diagonale inferiore):


,df_0,df_1,df_2,df_3,df_4,df_5,df_6
df_0,4143,,,,,,
df_1,0,847,,,,,
df_2,0,0,2222,,,,
df_3,1,0,0,4350,,,
df_4,3503,0,0,1,3504,,
df_5,838,0,0,0,745,840,
df_6,2567,0,0,1,2472,731,2597


In [13]:
# Lista dei dataframe e nomi, per comodità
dfs = [df_0, df_1, df_2, df_3, df_4, df_5, df_6]
df_names = [file_name_0, file_name_1, file_name_2, file_name_3, file_name_4, file_name_5, file_name_6]
df_code = ['df_0', 'df_1', 'df_2', 'df_3', 'df_4', 'df_5', 'df_6']

# Assicurati che le colonne RID e VISCODE siano presenti e che EXAMVISCODEDATE sia in formato datetime
for i, df in enumerate(dfs):
    if 'VISCODE' not in df.columns:
        print(f"AVVISO: '{df_names[i]}' non contiene colonna VISCODE")
    if 'RID' not in df.columns:
        print(f"AVVISO: '{df_names[i]}' non contiene colonna RID")

# Matrice di match per (RID, VISCODE)
n = len(dfs)
match_matrix = pd.DataFrame(0, index=df_code, columns=df_code)

for i in range(n):
    for j in range(n):
        # Ci assicuriamo di confrontare solo se entrambe le colonne esistono nei dati
        if {'RID', 'VISCODE'}.issubset(dfs[i].columns) and {'RID', 'VISCODE'}.issubset(dfs[j].columns):
            s1 = set(dfs[i][['RID', 'VISCODE']].drop_duplicates().itertuples(index=False, name=None))
            s2 = set(dfs[j][['RID', 'VISCODE']].drop_duplicates().itertuples(index=False, name=None))
            match_matrix.iloc[i, j] = len(s1 & s2)
        else:
            match_matrix.iloc[i, j] = None  # Indica colonne mancanti


display_match_matrix = match_matrix.copy()
mask = np.triu(np.ones(display_match_matrix.shape), k=1).astype(bool)
display_match_matrix = display_match_matrix.mask(mask, "")

print("Matrice del numero di righe con stessi RID (solo diagonale inferiore):")
display(display_match_matrix)





Matrice del numero di righe con stessi RID (solo diagonale inferiore):


,df_0,df_1,df_2,df_3,df_4,df_5,df_6
df_0,4087,,,,,,
df_1,0,845,,,,,
df_2,0,0,2220,,,,
df_3,5,0,0,4346,,,
df_4,3493,0,0,3,3501,,
df_5,838,0,0,0,744,840,
df_6,2567,0,0,2,2470,734,2597


## Approfondimento elementi in comune e non

In [13]:
# Per df_0, individua le righe (RID, EXAMDATE) che non sono presenti negli altri 5 df (df_1...df_5)
missing_rows_counts = []
non_matching_dfs = []

for k in range(1, 6):  # confronta df_0 con df_1...df_5
    # Seleziona le tuple (RID, EXAMDATE) per entrambi i df
    s0 = set(df_0[['RID', 'EXAMDATE']].drop_duplicates().itertuples(index=False, name=None)) if {'RID', 'EXAMDATE'}.issubset(df_0.columns) else set()
    sj = set(dfs[k][['RID', 'EXAMDATE']].drop_duplicates().itertuples(index=False, name=None)) if {'RID', 'EXAMDATE'}.issubset(dfs[k].columns) else set()
    both = s0 & sj
    not_in_other = s0 - sj
    opposite = sj - s0
    print(f"Numero di righe (RID, EXAMDATE) comuni a df_0 e {df_names[k]}:", len(both))
    print(f"Numero di righe (RID, EXAMDATE) di df_0 non presenti in {df_names[k]}:", len(not_in_other))
    print(f"Numero di righe (RID, EXAMDATE) di {df_names[k]} non presenti in df_0:", len(opposite), '\n')
    missing_rows_counts.append(len(not_in_other))
    # Crea un DataFrame con le sole righe di df_0 che non hanno (RID, EXAMDATE) in common con l'altro df
    df_0_nonmatch = df_0.set_index(['RID', 'EXAMDATE']).loc[list(not_in_other)].reset_index() if not_in_other else pd.DataFrame(columns=df_0.columns)
    non_matching_dfs.append(df_0_nonmatch)

# non_matching_dfs[i] contiene il DataFrame delle righe di df_0 che non sono nel corrispondente df_1...df_5 (in ordine)
# Ad esempio: non_matching_dfs[0] --> vs df_1; non_matching_dfs[1] --> vs df_2 ...


Numero di righe (RID, EXAMDATE) comuni a df_0 e UCSFFSX7_11Aug2025_04.csv: 0
Numero di righe (RID, EXAMDATE) di df_0 non presenti in UCSFFSX7_11Aug2025_04.csv: 4143
Numero di righe (RID, EXAMDATE) di UCSFFSX7_11Aug2025_04.csv non presenti in df_0: 847 

Numero di righe (RID, EXAMDATE) comuni a df_0 e UCSFFSX6_11Aug2025_04.csv: 0
Numero di righe (RID, EXAMDATE) di df_0 non presenti in UCSFFSX6_11Aug2025_04.csv: 4143
Numero di righe (RID, EXAMDATE) di UCSFFSX6_11Aug2025_04.csv non presenti in df_0: 2222 

Numero di righe (RID, EXAMDATE) comuni a df_0 e UCSFFSX51_11_08_19_11Aug2025_04.csv: 1
Numero di righe (RID, EXAMDATE) di df_0 non presenti in UCSFFSX51_11_08_19_11Aug2025_04.csv: 4142
Numero di righe (RID, EXAMDATE) di UCSFFSX51_11_08_19_11Aug2025_04.csv non presenti in df_0: 4349 

Numero di righe (RID, EXAMDATE) comuni a df_0 e UCSFFSL_02_01_16_11Aug2025_04.csv: 3503
Numero di righe (RID, EXAMDATE) di df_0 non presenti in UCSFFSL_02_01_16_11Aug2025_04.csv: 640
Numero di righe (RID, E

In [41]:
# Trova i valori di RID comuni tra df_0 e df_1
if 'RID' in df_0.columns and 'RID' in df_2.columns:
    rids_common = set(df_0['RID']).intersection(df_1['RID'])
    print(f"Numero di RID comuni tra df_0 e df_1: {len(rids_common)}")
    print("Esempio di RID comuni:", list(rids_common)[:10])
else:
    print("Una delle due tabelle non ha la colonna 'RID'")


Numero di RID comuni tra df_0 e df_1: 11
Esempio di RID comuni: [1280, 1155, 69, 1222, 679, 72, 934, 21, 89, 413]


In [47]:
df_0[df_0['RID']==1155]

,RID,VISCODE,VISIT_MONTH,EXAMDATE,STATUS,ICV%ICV,MidTemp%ICV,Fusiform%ICV,Ventricles%ICV,Entorhinal%ICV,Hippocampus%ICV
3431,1155,sc,0,2006-12-14,complete,100.0,1.322424,1.319898,2.333682,0.207905,0.510831
3432,1155,m06,6,2007-06-21,complete,100.0,1.314810,1.269171,2.311401,0.204450,0.486900
3433,1155,m12,13,2008-01-16,complete,100.0,1.299393,1.301833,2.419721,0.209712,0.495418
3434,1155,m18,20,2008-08-21,complete,100.0,1.296436,1.275503,2.468984,0.211572,0.504694
3435,1155,m24,27,2009-03-13,complete,100.0,1.317415,1.202766,2.602516,0.210100,0.487092
3436,1155,m36,37,2010-01-07,complete,100.0,1.321939,1.302928,2.562227,0.231491,0.499941
3437,1155,m48,49,2011-01-06,complete,100.0,1.263292,1.213318,2.735916,0.216149,0.491706


In [48]:
df_2[df_2['RID']==1155]

,COHORT,RID,VISCODE,VISIT_MONTH,EXAMDATE,STATUS,ICV%ICV,MidTemp%ICV,Fusiform%ICV,Ventricles%ICV,Entorhinal%ICV,Hippocampus%ICV
25,ADNI3,1155,m126,0,2017-04-24,complete,100.0,1.391028,1.318085,3.360564,0.258240,0.520478
26,ADNI3,1155,m138,12,2018-05-08,complete,100.0,1.419243,1.336539,3.509809,0.244513,0.499319
27,ADNI3,1155,m150,25,2019-05-16,complete,100.0,1.438455,1.314569,3.642221,0.249562,0.503682


# Inizio Merge
## Definizione di df_merge_0 e Gerarchia di DF da mergiare
Scegliere il file con numero maggiore di soggetti-visite e che ha più elementi con altri df.\
Quindi scegliere con che ordine unire gli altri df, suggerimento da quelli con nessuna/pochissime righe RID-EXAMDATE in comune con gli altri df, e quindi quelli con molte righe in comune a partire da quello con più righe in comune sia con df_merge_0 che con gli altri e quindi a seguire. Ma di persè il metodo è arbitrario quindi si può fare come si vuole.

In [25]:
df_merge_0 = df_0
to_merge_hierarchy = [df_2, df_1, df_3, df_4, df_6, df_5]
time_buffer = 30
i = 0

In [22]:
df_to_add = to_merge_hierarchy[3]


## Studio soggetti, soggetti-date, soggetti-visite in comune tra i 2 df identificati

In [23]:
df_to_add

,RID,VISCODE,VISIT_MONTH,EXAMDATE,STATUS,ICV%ICV,MidTemp%ICV,Fusiform%ICV,Ventricles%ICV,Entorhinal%ICV,Hippocampus%ICV
0,3,sc,0,2005-09-01,complete,100.0,0.939454,0.845425,3.737928,0.122404,0.270528
1,3,m06,6,2006-03-13,complete,100.0,0.929958,0.788752,3.931380,0.123634,0.261011
2,3,m12,12,2006-09-12,complete,100.0,0.912534,0.766722,3.993340,0.113981,0.263418
3,3,m24,24,2007-09-12,complete,100.0,0.856931,0.721753,4.330573,0.105704,0.251022
4,4,sc,0,2005-09-22,complete,100.0,1.163662,1.160506,2.289870,0.231208,0.380186
...,...,...,...,...,...,...,...,...,...,...,...
3499,1427,m18,19,2009-03-12,complete,100.0,1.391516,1.021851,1.398308,0.222350,0.471934
3500,1427,m24,24,2009-08-26,complete,100.0,1.455481,1.064123,1.304354,0.226921,0.471741
3501,1427,m48,48,2011-08-19,complete,100.0,1.470185,1.056177,1.503969,0.214258,0.473785
3502,1430,sc,0,2007-09-07,complete,100.0,1.087813,1.073505,1.861098,0.203600,0.311156


In [26]:
### Ripasso quanti soggetti, e soggetto-visita in comune hanno i 2 df

# Lista dei RID in comune tra i due DataFrame (ad esempio df_merge_0 e il primo da unire)
if 'RID' in df_merge_0.columns and 'RID' in df_to_add.columns:
    subj_in_common = set(df_merge_0['RID']).intersection(df_to_add['RID'])
    print(f"Numero di Soggetti in comune tra df_merge_0 e df da unire: {len(subj_in_common)}")
    print("Esempio di RID in comune:", list(subj_in_common)[:10])
    
    # lista RID-EXAMDATE (+-time_buffer) comuni
    if 'EXAMDATE' in df_merge_0.columns and 'EXAMDATE' in df_to_add.columns:
        subj_date_common_df = count_matches_with_buffer(df_merge_0, df_to_add, time_buffer)
        print(f"Numero di Soggetti in comune tra df_merge_0 e df da unire: {len(subj_date_common_df)}")
        print("Esempio di RID in comune:", subj_date_common_df.head())
    else:
        print("una delle 2 tabelle non ha la colonna 'EXAMDATE'")
    
    # lista RID-VISITCODE comuni
    if 'VISCODE' in df_merge_0.columns and 'VISCODE' in df_to_add.columns:
        subj_viscode_in_common = set(zip(df_merge_0['RID'], df_merge_0['VISCODE'])).intersection(set(zip(df_to_add['RID'], df_to_add['VISCODE'])))
        print(f"Numero di Soggetti in comune tra df_merge_0 e df da unire: {len(subj_viscode_in_common)}")
        print("Esempio di RID in comune:", list(subj_viscode_in_common)[:10])
    else:
        print("una delle 2 tabelle non ha la colonna 'VISCODE'")
        subj_viscode_in_common = []
else:
    print("Una delle due tabelle non ha la colonna 'RID'")

Numero di Soggetti in comune tra df_merge_0 e df da unire: 755
Esempio di RID in comune: [3, 4, 5, 6, 7, 8, 10, 14, 15, 16]


TypeError: Invalid comparison between dtype=timedelta64[ns] and int

In [14]:
# Trova tutte le coppie (RID, VISCODE) in df_merge_0 i cui (RID, EXAMDATE) corrispondono alle (RID, EXAMDATE_1) presenti in subj_date_common_df
if 'RID' in df_merge_0.columns and 'EXAMDATE' in df_merge_0.columns and 'VISCODE' in df_merge_0.columns:
    # Filtro df_merge_0 sulle tuple (RID, EXAMDATE) presenti in subj_date_common_df
    merge_keys = set(zip(subj_date_common_df['RID'], subj_date_common_df['EXAMDATE_1']))
    mask = df_merge_0.apply(lambda row: (row['RID'], row['EXAMDATE']) in merge_keys, axis=1)
    matched_df_merge = df_merge_0[mask]

    # Estrai le coppie (RID, VISCODE) dalla selezione
    rid_viscode_matched_list = set(zip(matched_df_merge['RID'], matched_df_merge['VISCODE']))
    
    # Confronta questa lista con subj_viscode_in_common
    subj_viscode_in_common_set = set(subj_viscode_in_common)
    not_in_matched = subj_viscode_in_common_set - rid_viscode_matched_list
    
    print(f"Numero di elementi di subj_viscode_in_common NON trovati tra le nuove coppie (RID, VISCODE): {len(not_in_matched)}")
    print("Esempi di coppie (RID, VISCODE) assenti:", list(not_in_matched)[:10])
else:
    print("df_merge_0 non ha tutte le colonne richieste ('RID', 'EXAMDATE', 'VISCODE')")

if len(not_in_matched) > 0:
    # Trova gli indici delle righe in df_merge_0 che corrispondono alle coppie in not_in_matched
    mask_not_in_matched = df_merge_0.apply(lambda row: (row['RID'], row['VISCODE']) in not_in_matched, axis=1)
    df_not_in_matched = df_merge_0[mask_not_in_matched]
    print("Indici delle righe in df_merge_0 corrispondenti alle coppie not_in_matched:")
    print(df_not_in_matched.index.tolist())
    print("Primi esempi delle righe:")
    print(df_not_in_matched.head())
else:
    print("Tutte le coppie (RID, VISCODE) di subj_viscode_in_common sono presenti tra le nuove coppie.")


NameError: name 'df_merge_0' is not defined

# altro

In [ ]:
# Seleziona solo le righe in cui TUTTI i volumi richiesti NON sono NaN
vol_cols = [
    "Ventricles%ICV",
    "Hippocampus%ICV",
    "Entorhinal%ICV",
    "Fusiform%ICV",
    "MidTemp%ICV",
    "ICV%ICV"
]
df_merge_vol = df_merge.dropna(subset=vol_cols, how="any").copy()

# Di che colonne identificative vogliamo il confronto?
# Ragionevolmente, 'RID' (subject id) -- eseguiamo il confronto su questo campo.

# Set di RIDs nei due dataframe
rids_merge_vol = set(df_merge_vol["RID"].unique())
rids_vol = set(df_vol["RID"].unique())

# 1) Quanti soggetti in comune hanno i due df
common_rids = rids_merge_vol & rids_vol
print("Numero di soggetti in comune tra df_merge_vol e df_vol:", len(common_rids))

# 2) Quanti sogg ha df_vol che df_merge_vol non ha
only_in_vol = rids_vol - rids_merge_vol
print("Numero di soggetti che sono in df_vol ma non in df_merge_vol:", len(only_in_vol))


In [ ]:
print('rows ADNIMERGE', len(df_merge))
print('rows df_vol', len(df_vol))
print('rows df_merge_vol', len(df_merge_vol))


In [ ]:
import pandas as pd

# Assicurati che le colonne EXAMDATE siano in formato datetime
df_vol['EXAMDATE'] = pd.to_datetime(df_vol['EXAMDATE'], errors='coerce')
df_merge_vol['EXAMDATE'] = pd.to_datetime(df_merge_vol['EXAMDATE'], errors='coerce')

# Per velocità, possiamo ordinare i dataframe
df_merge_vol_sorted = df_merge_vol.sort_values(['RID', 'EXAMDATE'])
df_vol_sorted = df_vol.sort_values(['RID', 'EXAMDATE'])

common_count = 0
only_in_vol_count = 0

idxs_in_common = []
idxs_only_in_vol = []

# Per ciascuna riga di df_vol, verifichiamo se esiste una riga dello stesso RID in df_merge_vol con EXAMDATE a +/- 15gg
for idx, row in df_vol_sorted.iterrows():
    rid = row['RID']
    examdate = row['EXAMDATE']
    sub_merge = df_merge_vol_sorted[df_merge_vol_sorted['RID'] == rid]
    # Trova se c'è almeno un examdate entro +/-15 giorni
    mask_in_range = sub_merge[
        (sub_merge['EXAMDATE'] - examdate).abs().dt.days <= 80
    ]
    if not mask_in_range.empty:
        common_count += 1
        idxs_in_common.append(idx)
    else:
        only_in_vol_count += 1
        idxs_only_in_vol.append(idx)

print(f"Numero di righe in comune tra df_vol e df_merge_vol (stesso RID e EXAMDATE a +/-15gg): {common_count}")
print(f"Numero di righe di df_vol che NON hanno match in df_merge_vol con stesso RID ed EXAMDATE a +/-15gg: {only_in_vol_count}")




In [ ]:
# Seleziona le righe di df_vol che NON hanno un match in df_merge_vol con stesso RID ed EXAMDATE a +/-15gg
df_only_in_vol = df_vol_sorted.loc[idxs_only_in_vol]
print("\nPrime 10 righe di df_vol che df_merge_vol non ha (confronto su RID + EXAMDATE +/-80gg):")
display(df_only_in_vol.head(10))

In [ ]:
import numpy as np
# Calcola quanti soggetti (RID) in df_only_in_vol sono anche presenti in df_merge_vol (indipendentemente dalla data)
unique_rid_only_in_vol = df_only_in_vol['RID'].unique()
unique_rid_merge_vol = df_merge_vol['RID'].unique()
unique_rid_merge = df_merge['RID'].unique()

count_rid_in_both = sum(np.isin(unique_rid_only_in_vol, unique_rid_merge_vol))
count_rid_in_original = sum(np.isin(unique_rid_only_in_vol, unique_rid_merge))
print(f"Numero di soggetti (RID) in df_only_in_vol che sono presenti anche in df_merge_vol: {count_rid_in_both} / {len(unique_rid_only_in_vol)}")
print(f"Numero di soggetti (RID) in df_only_in_vol che sono presenti anche in ADNIMERGE: {count_rid_in_original} / {len(unique_rid_only_in_vol)}")
# Trova i soggetti (RID) in df_only_in_vol che non sono mai presenti in ADNIMERGE (unique_rid_merge)
rids_not_in_adnimerge = [rid for rid in unique_rid_only_in_vol if rid not in unique_rid_merge]
print(f"Soggetti in df_only_in_vol CHE NON sono in ADNIMERGE (unique_rid_merge): {rids_not_in_adnimerge}")
print(f"Totale: {len(rids_not_in_adnimerge)}")


In [ ]:
df_out = df_vol[df_vol['RID'].isin(rids_not_in_adnimerge)]
len(df_out)


In [ ]:
df_merge_vol[df_merge_vol['RID']==123]